In [ ]:
%pip install ijson  openmeteo_requests requests_cache retry_requests

In [1]:


from minio import Minio
import ijson
import json
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
import os
import io
import openmeteo_requests
import requests_cache
from retry_requests import retry
import logging

# ── definition des zonz dz vacance  ───────────────────────────────────────────
def getConstanteZoneVAcance() :
    return {
            "Auvergne-Rhône-Alpes":       {"zone_a": True,  "zone_b": False, "zone_c": False},
            "Bourgogne-Franche-Comté":    {"zone_a": True,  "zone_b": False, "zone_c": False},
            "Bretagne":                    {"zone_a": False, "zone_b": True,  "zone_c": False},
            "Centre-Val de Loire":         {"zone_a": True,  "zone_b": False, "zone_c": False},
            "Corse":                       {"zone_a": True,  "zone_b": False, "zone_c": False},
            "Grand Est":                   {"zone_a": True,  "zone_b": False, "zone_c": False},
            "Hauts-de-France":             {"zone_a": False, "zone_b": True,  "zone_c": False},
            "Île-de-France":               {"zone_a": False, "zone_b": False, "zone_c": True},
            "Normandie":                   {"zone_a": False, "zone_b": True,  "zone_c": False},
            "Nouvelle-Aquitaine":          {"zone_a": False, "zone_b": True,  "zone_c": False},
            "Occitanie":                   {"zone_a": True,  "zone_b": False, "zone_c": False},
            "Pays de la Loire":            {"zone_a": False, "zone_b": True,  "zone_c": False},
            "Provence-Alpes-Côte d'Azur":  {"zone_a": True,  "zone_b": False, "zone_c": False},
        }

def getConstanteChefLieuRegion():
    return {
        "Auvergne-Rhône-Alpes":       {"ville": "Lyon",       "lat": 45.7640, "lon": 4.8357},
        "Bourgogne-Franche-Comté":    {"ville": "Dijon",      "lat": 47.3220, "lon": 5.0415},
        "Bretagne":                    {"ville": "Rennes",     "lat": 48.1173, "lon": -1.6778},
        "Centre-Val de Loire":         {"ville": "Orléans",    "lat": 47.9029, "lon": 1.9039},
        "Corse":                       {"ville": "Ajaccio",    "lat": 41.9267, "lon": 8.7369},
        "Grand Est":                   {"ville": "Strasbourg", "lat": 48.5734, "lon": 7.7521},
        "Hauts-de-France":             {"ville": "Lille",      "lat": 50.6292, "lon": 3.0573},
        "Île-de-France":               {"ville": "Paris",      "lat": 48.8566, "lon": 2.3522},
        "Normandie":                   {"ville": "Rouen",      "lat": 49.4432, "lon": 1.0999},
        "Nouvelle-Aquitaine":          {"ville": "Bordeaux",   "lat": 44.8378, "lon": -0.5792},
        "Occitanie":                   {"ville": "Toulouse",   "lat": 43.6047, "lon": 1.4442},
        "Pays de la Loire":            {"ville": "Nantes",     "lat": 47.2184, "lon": -1.5536},
        "Provence-Alpes-Côte d'Azur":  {"ville": "Marseille",  "lat": 43.2965, "lon": 5.3698},
    }
def getDropColonneDataset() :
    return [
        "code_region", "courbe_moyenne_ndeg1_ndeg2_wh", "courbe_moyenne_ndeg1_wh","jour_max_du_mois_0_1","semaine_max_du_mois_0_1",
        "courbe_moyenne_ndeg2_wh", "indice_representativite_courbe_ndeg1_ndeg2",
        "indice_representativite_courbe_ndeg1", "indice_representativite_courbe_ndeg2",
        "_id", "_score", "_i", "_rand"
        ]


# ── fonction connexion database ───────────────────────────────────────────
def getConnexionbdd():
    DB_USER = "datalake"
    DB_PASS = "datalake"
    DB_HOST = "postgres"
    DB_PORT = 5432
    DB_NAME = "datalake"
    
    return create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# ── fonction client bucket───────────────────────────────────────────
def getConnexionBucket() :
      
    return Minio(
    "minio:9000",
    access_key="admin",
    secret_key="password",
    secure=False)


# ── fonction creation connexion api meteo ───────────────────────────────────────────
def getConnexionMeteo() :
    # Setup the Open-Meteo API client with cache and retry on error
    cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
    retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
    return openmeteo_requests.Client(session = retry_session)


# ── fonction d'agrega de chaque paquet json enedis , packet de 50000 ───────────────────────────────────────────
# ── Elle commance pas droper les colonne  ───────────────────────────────────────────
# ── creer la colonne de reference date absolu yyyy-mm-dd ───────────────────────────────────────────
# ── merge avec le datafram meteo_vacance sur la clef date region ───────────────────────────────────────────
# ── puis inserte en base avec condition replace si c'est le premier packet sinon append on ajoute ───────────────────────────────────────────

def process_batch(buffer,isfiret,table_name,cols_to_drop,engine):
    df_enedis = pd.DataFrame(buffer)
    df_enedis = df_enedis.drop(columns=cols_to_drop)
    df_enedis["date"] = pd.to_datetime(df_enedis["horodate"]).dt.date
    # enrichissement
    df_enedis_all = df_enedis.merge(mete_vac_ville, on=["date", "region"], how="left")


    
    ifexite = "append" 
    if isfiret :
        # écriture base
        ifexite = "replace" 
        isfiret = False

   

    df_enedis_all.to_sql(
        table_name,
        engine,
        schema="public", 
        if_exists=ifexite,  # "replace" = supprime si existe, "append" = ajoute
        index=False
    )
    print(f"record {df_enedis.shape }")
    return isfiret

# ── Requête météo pour chaque ville ───────────────────────────────────────────
def fetch_meteo_ville(ville: str, lat: float, lon: float, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Récupère température, humidité, précipitations horaires
    pour une ville via Open-Meteo Archive API.
    Retourne un DataFrame journalier (moyenne/somme du jour).
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   lat,
        "longitude":  lon,
        "start_date": start_date,
        "end_date":   end_date,
        "hourly": [
            "relative_humidity_2m",
            "precipitation",
            "temperature_2m"
        ],
    }

    logging.info(f"🌤️  Requête météo pour {ville} ({lat}, {lon})")

    responses = openmeteo.weather_api(url, params=params)
    response  = responses[0]
    hourly    = response.Hourly()

    # ── Extraction des variables (ordre = ordre dans params["hourly"]) ──
    hourly_relative_humidity_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_precipitation        = hourly.Variables(1).ValuesAsNumpy()
    hourly_temperature_2m       = hourly.Variables(2).ValuesAsNumpy()

    # ── Construction du DataFrame horaire ────────────────────────────────
    df_hourly = pd.DataFrame({
        "date": pd.date_range(
            start     = pd.to_datetime(hourly.Time(),    unit="s", utc=True),
            end       = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq      = pd.Timedelta(seconds=hourly.Interval()),
            inclusive = "left"
        ),
        "relative_humidity_2m": hourly_relative_humidity_2m,
        "precipitation":        hourly_precipitation,
        "temperature_2m":       hourly_temperature_2m,
    })

    # ── Passage en heure locale (UTC → Europe/Paris) ──────────────────────
    df_hourly["date"] = df_hourly["date"].dt.tz_convert("Europe/Paris")

    # ── Agrégation journalière ────────────────────────────────────────────
    # température  → moyenne du jour
    # humidité     → moyenne du jour
    # précipitation → somme du jour
    df_daily = (
        df_hourly
        .groupby(df_hourly["date"].dt.date)
        .agg(
            temperature_2m_mean    = ("temperature_2m",       "mean"),
            relative_humidity_mean = ("relative_humidity_2m", "mean"),
            precipitation_sum      = ("precipitation",        "sum"),
        )
        .reset_index()
        .rename(columns={"date": "date"})
    )

    df_daily["date"]  = pd.to_datetime(df_daily["date"])
    df_daily["ville"] = ville

    return df_daily
 


print("start for json")
#date beetwen pour meteo
#celle-ci sont determiner par de dataset enedis
date_min = None
date_max = None

#client minio 
client = getConnexionBucket()

# Setup the Open-Meteo API client with cache and retry on error
openmeteo = getConnexionMeteo()

#premier parcours du dataset json pour recuperer les dates
for enedis in ijson.items(client.get_object("datalake","api_conso_electrique_agregats_30min_points_soutirage.json"),"item") :
    #print(f"{enedis}")
    d=datetime.fromisoformat(enedis["horodate"])
    #print(f"{d}")
    if date_min is None or d < date_min :
        date_min = d
    if date_max is None or d > date_max :
        date_max = d
   

#df_enedis["date"] = pd.to_datetime(df_enedis["horodate"]).dt.date
print(f'min {date_min.strftime("%Y-%m-%d")} max  {date_max.strftime("%Y-%m-%d")}')
   
 
print("Recupertion du ficher clandrier dans le bucker")
obj_parquet = client.get_object("datalake", "calendrier_vacances_scolaires_france.parquet")
obj_parquet_vac = pd.read_parquet(io.BytesIO(obj_parquet.read()))
print(f"obj_parquet_vac {len(obj_parquet_vac)}")

# ── Régions / Capitales / Zones ───────────────────────────────────────────────
capitales = getConstanteChefLieuRegion()
zones_vacances = getConstanteZoneVAcance()

rows = []
for region, info in capitales.items():
    rows.append({
        "region": region,
        "ville":  info["ville"],
        "lat":    info["lat"],
        "lon":    info["lon"],
        "zone_a": zones_vacances[region]["zone_a"],
        "zone_b": zones_vacances[region]["zone_b"],
        "zone_c": zones_vacances[region]["zone_c"],
    })

df_region = pd.DataFrame(rows)
print(f"df_region ",df_region.shape)
villes_df = df_region[["ville", "lat", "lon"]].drop_duplicates()
print(f"villes_df ",villes_df.shape)

meteo_frames = []
for _, row in villes_df.iterrows():
    df_meteo = fetch_meteo_ville(
        ville      = row["ville"],
        lat        = row["lat"],
        lon        = row["lon"],
        start_date = date_min.strftime('%Y-%m-%d'),
        end_date   = date_max.strftime('%Y-%m-%d'),
    )
    meteo_frames.append(df_meteo)

df_meteo_all = pd.concat(meteo_frames, ignore_index=True)

print("df_meteo_all  :", df_meteo_all.shape)
print('-------------------------------')
print(df_meteo_all.head())


df_meteo_all["date"] = pd.to_datetime(df_meteo_all["date"])
obj_parquet_vac["date"] = pd.to_datetime(obj_parquet_vac["date"])
mete_vac = df_meteo_all.merge(obj_parquet_vac, on="date", how="left")
print("mete_vac  :", mete_vac.shape)
print('-------------------------------')
print(mete_vac.head())
print('-------------------------------')
mete_vac_ville = mete_vac.merge(df_region, on="ville", how="left")
mete_vac_ville["date"] = pd.to_datetime(mete_vac_ville["date"]).dt.date
print(f"mete_vac_ville :", mete_vac_ville.shape)
print('-------------------------------')


#definition des variable
#contenir les packet lu
buffer = []
#taille du buffer
batch_size = 50_000  # à ajuster en fonction de la ram
#determine le premier passage
isfiret = True
#table de la bdd
table_name = "enedisent"  # nom de la table à créer
#constante des colonne
cols_to_drop = getDropColonneDataset()
#appele a la connexion bdd
engine = getConnexionbdd()
#lecture du json a partir du bucket
response = client.get_object("datalake", "api_conso_electrique_agregats_30min_points_soutirage_all.json")
objects = ijson.items(response, "item")

#traitement par packet du fichier
for obj in objects:
    buffer.append(obj)

    if len(buffer) >= batch_size:
        isfiret =  process_batch(buffer,isfiret,table_name,cols_to_drop,engine)
        buffer = []

# dernier batch si le buffer n'est pas de 50000
if buffer:
    process_batch(buffer ,isfiret,table_name,cols_to_drop,engine )

#close des stream
response.close()
response.release_conn()

ModuleNotFoundError: No module named 'minio'